In [1]:
import pandas as pd
import subprocess
import scipy.stats as stats
import altair as alt
import numpy as np
from pathlib import Path
import sys
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

# ClinVar summary pre-preocessing

In [17]:
clinvar_file='/net/bbi/vol1//home/ivanw314/work/20260428_SpliceAI_ClinVar_benchmark_supporting_files/variant_summary_2025-01.txt' #ClinVar Jan 2025 variant summary

In [18]:
original_clinvar_df = pd.read_csv(clinvar_file, sep='\t') #Read file

/tmp/16339303.1.shendure-login.q/ipykernel_91204/1145797189.py:1: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  original_clinvar_df = pd.read_csv(clinvar_file, sep='\t') #Read file


In [5]:
clinvar_df = original_clinvar_df[['Type','Name', 'GeneSymbol', 'Assembly', 'Chromosome', 'Start', 'Stop', 'ClinicalSignificance', 'ReviewStatus', 'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF']] #Grab desired columns
clinvar_df = clinvar_df.loc[(clinvar_df['Assembly']=='GRCh38') & (clinvar_df['Type']== 'single nucleotide variant') & (clinvar_df['Name'].str.contains(':'))].copy() #Filter for hg19 and SNVs only
filtered_full_clinvar_df = clinvar_df[clinvar_df['ReviewStatus'].isin(['criteria provided', 'multiple submitters, no conflicts', 'criteria provided, conflicting classifications', 'criteria provided, single submitter', 'reviewed by expert panel', 'practice guideline'])].copy() #Filter for 1-star plus

In [6]:
filtered_full_clinvar_df.head()

,Type,Name,GeneSymbol,Assembly,Chromosome,Start,Stop,ClinicalSignificance,ReviewStatus,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF
11,single nucleotide variant,NM_025152.3(NUBPL):c.166G>A (p.Gly56Arg),NUBPL,GRCh38,14,31562125,31562125,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",31562125,G,A
17,single nucleotide variant,NM_000410.4(HFE):c.193A>T (p.Ser65Cys),HFE,GRCh38,6,26090957,26090957,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",26090957,A,T
19,single nucleotide variant,NM_000410.4(HFE):c.314T>C (p.Ile105Thr),HFE,GRCh38,6,26091078,26091078,Uncertain significance,"criteria provided, single submitter",26091078,T,C
21,single nucleotide variant,NM_000410.4(HFE):c.277G>C (p.Gly93Arg),HFE,GRCh38,6,26091041,26091041,Uncertain significance,"criteria provided, single submitter",26091041,G,C
23,single nucleotide variant,NM_000410.4(HFE):c.892+48G>A,HFE,GRCh38,6,26093008,26093008,Benign,"criteria provided, single submitter",26093008,G,A


In [19]:
filtered_full_clinvar_df['hgvs_c']=filtered_full_clinvar_df['Name'].transform(lambda x: x.split(':')[1]) #Pull hgvs_c from Name column
filtered_full_clinvar_df['aa_change']=filtered_full_clinvar_df['hgvs_c'].transform(lambda x: x.split(' ')[1][1:-1] if 'p.' in x else np.nan) #Gets amino acid change if applicable
filtered_full_clinvar_df['pos_id']=filtered_full_clinvar_df['GeneSymbol'] + ':' + filtered_full_clinvar_df['Start'].astype(str) + ':' + filtered_full_clinvar_df['AlternateAlleleVCF'] #Make variant ID field
filtered_full_clinvar_df['broad_consequence'] = filtered_full_clinvar_df['aa_change'].transform( 
    lambda x: 'non_coding' if (pd.isna(x) or 'p.' not in x)
              else 'synonymous' if '=' in x
              else 'non_synonymous'
) #broadly classsifies molecular consequence

filtered_full_clinvar_df['intronic_dist'] = filtered_full_clinvar_df['hgvs_c'].str.extract(r'[+-](\d+)') #Extracts distance from coding sequence
filtered_full_clinvar_df.head()

,Type,Name,GeneSymbol,Assembly,Chromosome,Start,Stop,ClinicalSignificance,ReviewStatus,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,hgvs_c,aa_change,pos_id,broad_consequence,intronic_dist
11,single nucleotide variant,NM_025152.3(NUBPL):c.166G>A (p.Gly56Arg),NUBPL,GRCh38,14,31562125,31562125,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",31562125,G,A,c.166G>A (p.Gly56Arg),p.Gly56Arg,NUBPL:31562125:A,non_synonymous,NaN
17,single nucleotide variant,NM_000410.4(HFE):c.193A>T (p.Ser65Cys),HFE,GRCh38,6,26090957,26090957,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",26090957,A,T,c.193A>T (p.Ser65Cys),p.Ser65Cys,HFE:26090957:T,non_synonymous,NaN
19,single nucleotide variant,NM_000410.4(HFE):c.314T>C (p.Ile105Thr),HFE,GRCh38,6,26091078,26091078,Uncertain significance,"criteria provided, single submitter",26091078,T,C,c.314T>C (p.Ile105Thr),p.Ile105Thr,HFE:26091078:C,non_synonymous,NaN
21,single nucleotide variant,NM_000410.4(HFE):c.277G>C (p.Gly93Arg),HFE,GRCh38,6,26091041,26091041,Uncertain significance,"criteria provided, single submitter",26091041,G,C,c.277G>C (p.Gly93Arg),p.Gly93Arg,HFE:26091041:C,non_synonymous,NaN
23,single nucleotide variant,NM_000410.4(HFE):c.892+48G>A,HFE,GRCh38,6,26093008,26093008,Benign,"criteria provided, single submitter",26093008,G,A,c.892+48G>A,NaN,HFE:26093008:A,non_coding,48


In [20]:
syn_noncoding_clinvar_df = filtered_full_clinvar_df[filtered_full_clinvar_df['broad_consequence'].isin(['synonymous', 'non_coding'])].copy() #Gets generally synonymous and non-coding variants (includes intronic) mostly
syn_noncoding_clinvar_df = syn_noncoding_clinvar_df[syn_noncoding_clinvar_df['ClinicalSignificance'].isin(['Benign', 'Pathogenic','Likely pathogenic', 'Pathogenic; drug response', 'Likely pathogenic; drug response', 
                                                                                                           'Likely benign','Benign/Likely benign', 'risk factor','Pathogenic/Likely pathogenic'])].copy() #Filters out VUS

In [21]:
#Renames ClinVar consequences
syn_noncoding_clinvar_df.loc[syn_noncoding_clinvar_df['ClinicalSignificance'].str.contains('enign'), 'ClinicalSignificance'] = 'BLB'
syn_noncoding_clinvar_df.loc[syn_noncoding_clinvar_df['ClinicalSignificance'].str.contains('hogenic'), 'ClinicalSignificance'] = 'PLP'

for_vcf = syn_noncoding_clinvar_df.rename(columns = {'Chromosome': 'chrom', 
                                                     'PositionVCF': 'pos',
                                                     'ReferenceAlleleVCF': 'ref',
                                                     'AlternateAlleleVCF': 'alt'
                                                    }
                                         ) #Renames clinvar for VCF building

for_vcf = for_vcf[~for_vcf['chrom'].isin(['MT', 'Un'])] #Drops other chromosomes

#Some error checking and filtering for weird rows
for_vcf=for_vcf[for_vcf['pos']>0]
for_vcf=for_vcf[for_vcf['alt']!=for_vcf['ref']]
for_vcf = for_vcf.sort_values(["chrom", "pos"])

plp_vcf_df = for_vcf[for_vcf['ClinicalSignificance']=='PLP']

In [22]:
#Saves VCF if needed
save_vcf=False
if save_vcf:
    with open('/net/bbi/vol1/home/ivanw314/work/20260428_SpliceAI_ClinVar_benchmark_supporting_files/20260429_clinvar_vcf.vcf', 'w') as f:
        f.write("##fileformat=VCFv4.3\n")
        f.write("##reference=GRCh38\n")
        f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
        for _, row in for_vcf.iterrows():
            f.write(
                f"{row['chrom']}\t{row['pos']}\t.\t"
                f"{row['ref'].upper()}\t{row['alt'].upper()}\t.\tPASS\t.\n"
            )

# Analysis Start

In [23]:
vcf_path = '/net/bbi/vol1//home/ivanw314/work/20260428_SpliceAI_ClinVar_benchmark_supporting_files/20260429_clinvar_vcf_plp_vep_output.txt'

In [49]:
vep_df = pd.read_csv(vcf_path, sep='\t')
vep_df['pos']=vep_df['Location'].transform(lambda x: int(x.split(':')[1].split('-')[0]))
vep_df['pos_id']=vep_df['SYMBOL'] + ':' + vep_df['pos'].astype(str) + ':' +  vep_df['Allele']
vep_df['first_consequence']=vep_df['Consequence'].transform(lambda x: x.split(',')[0])
vep_df['maxSpliceAI'] = vep_df[['SpliceAI_pred_DS_AG', 'SpliceAI_pred_DS_AL', 'SpliceAI_pred_DS_DG', 'SpliceAI_pred_DS_DL']].max(axis=1)

vep_df = vep_df[['SYMBOL', 'Consequence', 'first_consequence', 'pos', 'pos_id', 'maxSpliceAI']]

vep_df = vep_df.rename(columns={'SYMBOL':'GeneSymbol'})
vep_df.head()

,GeneSymbol,Consequence,first_consequence,pos,pos_id,maxSpliceAI
0,AGRN,splice_acceptor_variant,splice_acceptor_variant,1035275,AGRN:1035275:G,1.00
1,AGRN,splice_donor_variant,splice_donor_variant,1043733,AGRN:1043733:T,1.00
2,AGRN,splice_acceptor_variant,splice_acceptor_variant,1043822,AGRN:1043822:A,0.94
3,AGRN,splice_donor_variant,splice_donor_variant,1045278,AGRN:1045278:T,1.00
4,AGRN,splice_donor_variant,splice_donor_variant,1046736,AGRN:1046736:A,0.94


In [66]:
vep_df_filtered = vep_df[(vep_df['Consequence'].str.contains('intron_variant')) | (vep_df['Consequence'].str.contains('synonymous'))] #Filters for any variant whose annotation contains intron or synonymous. This may include variants with splice region
vep_df_strict_filtered=vep_df[vep_df['first_consequence'].isin(['intron_variant', 'synonymous_variant'])] #Filters for pure intronic and synonymous variants only

dfs_for_analysis = {'all': vep_df_filtered,
                    'strict': vep_df_strict_filtered
                   }

In [68]:
def histogram(df, filter_type): #Histogram helper function
    
    splice_ai_dist = alt.Chart(df).mark_bar().encode(
        x=alt.X('maxSpliceAI:Q',
                axis=alt.Axis(title = 'Max SpliceAI'),
                bin=True),
        y=alt.Y('count():Q',
                axis=alt.Axis(title= '# Variants')
               )
    ).facet('first_consequence:N').properties(title=f'{filter_type} Intronic/Synonymous Variants SpliceAI')

    splice_ai_dist.display()


In [69]:
for key in dfs_for_analysis.keys():
    df_for_analysis = dfs_for_analysis[key]

    final_df = pd.merge(plp_vcf_df, df_for_analysis, on=['pos_id', 'pos', 'GeneSymbol'], how='inner')[['Name', 'GeneSymbol', 'chrom', 'ClinicalSignificance', 'ReviewStatus', 'pos', 'ref', 'alt', 'hgvs_c', 'aa_change', 'pos_id', 'Consequence','first_consequence', 'maxSpliceAI']]

    histogram(final_df, key)

alt.FacetChart(...)

alt.FacetChart(...)